# 3 - Category classification: what routes the ticket?

**Question.** Predict the routing category from the ticket text, to cut the
reassignments that misrouting generates. Does an LLM beat a linear model over
n-grams, and by enough to pay for itself?

| Approach | Representation | Classifier |
|---|---|---|
| `majority` | - | most frequent class |
| `nb` | word+char TF-IDF | Complement Naive Bayes |
| `logreg` | word+char TF-IDF | Logistic regression |
| `svm` | word+char TF-IDF | Calibrated LinearSVC |
| `embed_lr` | Sentence embeddings | Logistic regression |
| `llm` | - | Claude, constrained to the taxonomy |

**Every dependency is mandatory.** The embedding and LLM arms used to sit
behind optional extras and were skipped when absent. They are hard
requirements now, so the comparison cannot quietly lose a competitor.

The **one** thing an install cannot provide is API credentials and a budget.
`RUN_LLM` is therefore a *spend* gate, not a capability gate: section 4 prices
the run for free and you decide. Nothing here spends money without that flag.

### Three rules this notebook enforces

**`majority` is in the table on purpose.** A macro-F1 that does not clearly
beat the most-frequent-class baseline means the pipeline is broken, not that
the task is hard.

**Macro-F1, never accuracy.** Password resets outnumber database incidents by
two orders of magnitude. Accuracy rewards a model for the majority class and
says nothing about the small families where misrouting costs time.

**Temporal split, not random.** Train on the past, test on the most recent 20%.
Ticket vocabulary drifts, so a random split flatters every model equally and
predicts nothing about next month. Section 7 measures the gap, which *is* the
drift estimate.

```bash
uv sync
uv run itsm data build
export ANTHROPIC_API_KEY=...        # or `ant auth login`; only for the llm arm
```

### Runtime

The TF-IDF arms are seconds. `embed_lr` is the slow one - it encodes the whole
training set - and the encoder is cached process-wide, so the drift refit in
section 7 reuses the weights rather than loading them again. Expect a couple of
minutes on CPU end to end, plus the first-run model download.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

from itsm.config import get_settings
from itsm.models.classifier import TicketClassifier
from itsm.pipelines.anonymise import load_raw_tickets

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 40)

settings = get_settings()
TEXT_COLUMN = 'designation'   # the short title - what a triage UI has at submit
LABEL_COLUMN = 'category'
TEST_SIZE = 0.2

# Spend gate for the LLM arm. The package is installed; the budget is yours.
RUN_LLM = False
LLM_TEST_CAP = 300            # tickets sent to the API
LLM_MODEL = 'claude-opus-5'
LLM_BATCH = True              # Batch API: half price, minutes instead of seconds

## 1. The temporal split

Categories absent from the training window are dropped from the test set rather
than counted as free errors: a class the model has never seen is a taxonomy
change, not a model failure.

In [ ]:
tickets = load_raw_tickets(settings).dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
ordered = tickets.sort_values('opened_at')
cut = int(len(ordered) * (1 - TEST_SIZE))
train, test = ordered.iloc[:cut], ordered.iloc[cut:]

seen = set(train[LABEL_COLUMN])
dropped = int((~test[LABEL_COLUMN].isin(seen)).sum())
test = test[test[LABEL_COLUMN].isin(seen)]

x_train = train[TEXT_COLUMN].astype(str).tolist()
y_train = train[LABEL_COLUMN].astype(str).tolist()
x_test = test[TEXT_COLUMN].astype(str).tolist()
y_test = test[LABEL_COLUMN].astype(str).tolist()
LABELS = sorted(seen)

print(f'{len(x_train)} train / {len(x_test)} test, {len(LABELS)} categories')
print(f'{dropped} test rows dropped: class absent from the training window')

distribution = pd.Series(y_train).value_counts().to_frame('train')
distribution['test'] = pd.Series(y_test).value_counts()
distribution['train share'] = (distribution['train'] / len(y_train)).round(4)
distribution

## 2. The offline arms

Same class, different estimator, so the representation and preprocessing are
identical and only the model differs. That is what makes the column-to-column
comparison mean anything.

In [ ]:
ARMS = {
    'majority': ('majority', 'tfidf'),
    'nb': ('complement_nb', 'tfidf'),
    'logreg': ('logreg', 'tfidf'),
    'svm': ('linear_svc', 'tfidf'),
    'embed_lr': ('logreg', 'embeddings'),
}

# Written down before the run.
EXPECTED = {
    'majority': 'macro-F1 near 1/n_classes - the smoke test',
    'nb': 'decent and instant; loses to the discriminative models',
    'logreg': 'close to the SVM, better-behaved probabilities',
    'svm': 'the incumbent; seconds to train, inspectable n-grams',
    'embed_lr': 'better on paraphrase, worse on error codes and product names',
}


def score(y_true, y_pred):
    return {
        'macro F1': round(float(f1_score(y_true, y_pred, labels=LABELS, average='macro',
                                         zero_division=0)), 4),
        'balanced accuracy': round(float(balanced_accuracy_score(y_true, y_pred)), 4),
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 4),
    }


models, predictions, scores = {}, {}, {}
for name, (model, backend) in ARMS.items():
    started = time.perf_counter()
    classifier = TicketClassifier(model=model, backend=backend).fit(x_train, y_train)
    train_seconds = time.perf_counter() - started

    started = time.perf_counter()
    predicted = classifier.predict(x_test)
    predict_seconds = time.perf_counter() - started

    models[name] = classifier
    predictions[name] = predicted
    scores[name] = score(y_test, predicted)
    scores[name]['train seconds'] = round(train_seconds, 1)
    scores[name]['ms per ticket'] = round(1000 * predict_seconds / len(x_test), 2)
    scores[name]['expected'] = EXPECTED[name]
    print(f'{name:10s} macro F1 {scores[name]["macro F1"]:.4f}  fit {train_seconds:5.1f}s')

## 3. Offline results

In [ ]:
pd.DataFrame(scores)

## 4. The LLM arm - price it before running it

`count_tokens` is free, so the estimate below always runs. Two things make this
a fair comparison rather than a demo:

* the answer is **constrained to the taxonomy** by a JSON schema, so the error
  rate measures the model and not a regex failing to find a label in prose;
* it is asked for a **confidence**, because the coverage curve in section 6 is
  what a service desk actually buys, and an approach that cannot appear on that
  curve is not comparable on the decisive axis.

Self-reported confidence is **not** calibrated probability. Section 6 shows
whether it separates right from wrong at all.

Two runtime choices, both cost rather than speed: `effort: low` because
classification is not an intelligence-sensitive task, and the system prompt is
cached since it is byte-identical across every ticket.

In [ ]:
import anthropic

SYSTEM = (
    'You are an IT service-desk triage assistant. You classify incoming tickets '
    'into exactly one category from a fixed taxonomy.\n\n'
    'Tickets are in French or English, often terse, sometimes with typos. Personal '
    'data has already been replaced by tags such as <PERSON>; treat those as opaque '
    'placeholders and never let them influence the category.\n\n'
    'Report a confidence between 0 and 1 reflecting how certain you are. Use low '
    'values when the ticket is genuinely ambiguous: low-confidence tickets go to a '
    'human instead of being auto-routed, so an honest low confidence is more useful '
    'than a confident guess.'
)

SCHEMA = {
    'type': 'json_schema',
    'schema': {
        'type': 'object',
        'properties': {
            'label': {'type': 'string', 'enum': LABELS},
            'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        },
        'required': ['label', 'confidence'],
        'additionalProperties': False,
    },
}

# USD per million tokens, input/output. Verify against current pricing before
# quoting a figure to anyone.
PRICING = {'claude-opus-5': (5.0, 25.0), 'claude-sonnet-5': (2.0, 10.0),
           'claude-haiku-4-5': (1.0, 5.0)}


def llm_request(text):
    return {
        'model': LLM_MODEL,
        'max_tokens': 256,
        'system': [{'type': 'text', 'text': SYSTEM, 'cache_control': {'type': 'ephemeral'}}],
        'output_config': {'format': SCHEMA, 'effort': 'low'},
        'messages': [{'role': 'user', 'content': 'Ticket:\n' + (text or '')}],
    }


llm_test = x_test[:LLM_TEST_CAP]
llm_gold = y_test[:LLM_TEST_CAP]

# A bare client resolves ANTHROPIC_API_KEY, ANTHROPIC_AUTH_TOKEN, or an
# `ant auth login` profile - an unset env var is not proof of no credentials.
client = anthropic.Anthropic()
probe = llm_test[:20]
counted = sum(
    client.messages.count_tokens(
        model=LLM_MODEL,
        system=llm_request(t)['system'],
        messages=llm_request(t)['messages'],
    ).input_tokens
    for t in probe
)
mean_input = counted / len(probe)
rate_in, rate_out = PRICING[LLM_MODEL]
estimate = (mean_input * rate_in + 30 * rate_out) / 1e6 * len(llm_test)
if LLM_BATCH:
    estimate *= 0.5

print(f'{len(llm_test)} tickets, ~{mean_input:.0f} input tokens each')
print(f'estimated cost: ${estimate:.4f} ({LLM_MODEL}, batch={LLM_BATCH})')
print('Set RUN_LLM = True in cell 2 to spend this.' if not RUN_LLM else 'RUN_LLM is on.')

In [ ]:
llm_pred, llm_conf, llm_usage = None, None, {}

if RUN_LLM:
    tokens_in, tokens_out, failures = 0, 0, 0
    started = time.perf_counter()

    if LLM_BATCH:
        # The Batch API is the right path for an offline benchmark: nothing is
        # waiting on the answer, and the same run costs half. Results come back
        # in arbitrary order, so they are re-aligned by custom_id.
        requests = [
            anthropic.types.messages.batch_create_params.Request(
                custom_id='t-' + str(i),
                params=anthropic.types.MessageCreateParamsNonStreaming(**llm_request(t)),
            )
            for i, t in enumerate(llm_test)
        ]
        batch = client.messages.batches.create(requests=requests)
        print('batch', batch.id, 'submitted -', len(requests), 'requests')
        while client.messages.batches.retrieve(batch.id).processing_status != 'ended':
            time.sleep(20)

        llm_pred = [LABELS[0]] * len(llm_test)
        llm_conf = [0.0] * len(llm_test)
        for entry in client.messages.batches.results(batch.id):
            index = int(entry.custom_id.split('-')[1])
            if entry.result.type != 'succeeded':
                failures += 1
                continue
            message = entry.result.message
            tokens_in += message.usage.input_tokens
            tokens_out += message.usage.output_tokens
            payload = json.loads(next(b.text for b in message.content if b.type == 'text'))
            llm_pred[index] = payload['label']
            llm_conf[index] = float(payload.get('confidence', 0.0))
    else:
        llm_pred, llm_conf = [], []
        for text in llm_test:
            try:
                response = client.messages.create(**llm_request(text))
                tokens_in += response.usage.input_tokens
                tokens_out += response.usage.output_tokens
                payload = json.loads(next(b.text for b in response.content if b.type == 'text'))
                llm_pred.append(payload['label'])
                llm_conf.append(float(payload.get('confidence', 0.0)))
            except anthropic.APIError as error:
                # One failure must not lose the run; it is counted, not hidden.
                failures += 1
                llm_pred.append(LABELS[0])
                llm_conf.append(0.0)

    elapsed = time.perf_counter() - started
    cost = (tokens_in * rate_in + tokens_out * rate_out) / 1e6 * (0.5 if LLM_BATCH else 1.0)
    llm_usage = {
        'USD (this run)': round(cost, 4),
        'USD per 1k tickets': round(1000 * cost / len(llm_test), 4),
        'failed calls': failures,
        'ms per ticket': round(1000 * elapsed / len(llm_test), 1),
    }
    print(llm_usage)
else:
    print('LLM arm not run - RUN_LLM is False.')

### Like for like

The LLM saw only the first `LLM_TEST_CAP` test tickets, so the offline arms are
re-scored on **exactly that subset**. Comparing a model on 300 tickets with one
on 1500 is not a comparison.

In [ ]:
subset = {name: score(llm_gold, list(pred)[:LLM_TEST_CAP]) for name, pred in predictions.items()}
if llm_pred is not None:
    subset['llm'] = score(llm_gold, llm_pred)
    subset['llm'].update(llm_usage)

pd.DataFrame(subset)

## 5. Which classes fail

A macro-F1 moves because two or three families are broken. The table says
which - and two categories that confuse each other **symmetrically** usually
mean the taxonomy is wrong, not the model. That is worth more than a tuning
round.

In [ ]:
BEST = max(scores, key=lambda name: scores[name]['macro F1'])
print('best offline arm:', BEST)

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, predictions[BEST], labels=LABELS, zero_division=0
)
per_class = pd.DataFrame(
    {'precision': precision, 'recall': recall, 'f1': f1, 'support': support}, index=LABELS
).sort_values('f1')
per_class.round(3).head(8)

In [ ]:
worst = per_class.index[:4].tolist()
matrix = pd.DataFrame(
    confusion_matrix(y_test, predictions[BEST], labels=LABELS, normalize='true'),
    index=LABELS,
    columns=LABELS,
)
matrix.loc[worst].round(2)

In [ ]:
# Why the linear model is defensible beyond its score: it can be asked what it
# learned. An LLM cannot produce this table, and for a control that routes real
# tickets that is not a footnote.
if hasattr(models[BEST], 'top_features_for'):
    for label in worst[:3]:
        terms = models[BEST].top_features_for(label, n=8)
        print(label, '->', ', '.join(term for term, _ in terms))

## 6. The coverage curve - the decision-making metric

**This is what a service desk buys.** Not *how accurate is the model* but *what
share of tickets can we auto-route at 95% accuracy, and how many still need a
human?* Routing is automated above a confidence threshold; below it the ticket
abstains to human triage with the top-3 proposed.

The curve sets that threshold, which is why calibration matters more than the
last point of macro-F1.

In [ ]:
def coverage_curve(y_true, y_pred, confidence,
                   thresholds=(0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95)):
    truth, pred = np.asarray(y_true), np.asarray(y_pred)
    conf = np.asarray(confidence, dtype=float)
    rows = []
    for threshold in thresholds:
        mask = conf >= threshold
        covered = int(mask.sum())
        rows.append(
            {
                'threshold': threshold,
                'covered': covered,
                'coverage': round(covered / len(truth), 4),
                'accuracy above threshold': (
                    round(float((truth[mask] == pred[mask]).mean()), 4) if covered else None
                ),
            }
        )
    return pd.DataFrame(rows)


confidence = models[BEST].predict_proba(x_test).max(axis=1)
curve = coverage_curve(y_test, predictions[BEST], confidence)
print(BEST + ' (calibrated probability):')
display(curve)

if llm_conf:
    print('llm (self-reported confidence - not calibrated probability):')
    display(coverage_curve(llm_gold, llm_pred, llm_conf))

## 7. The drift estimate

The same arm on a random split instead of a temporal one. The **gap between the
two is the drift estimate**: how much of the model's apparent quality is
separability that will not survive next quarter's vocabulary.

In [ ]:
from sklearn.model_selection import train_test_split

frame = tickets[tickets[LABEL_COLUMN].isin(LABELS)]
s_train, s_test = train_test_split(
    frame, test_size=TEST_SIZE, random_state=42, stratify=frame[LABEL_COLUMN]
)
# The encoder is cached process-wide, so an `embeddings` refit here reloads
# nothing - it only re-encodes.
refit = TicketClassifier(model=ARMS[BEST][0], backend=ARMS[BEST][1]).fit(
    s_train[TEXT_COLUMN].astype(str).tolist(), s_train[LABEL_COLUMN].astype(str).tolist()
)

drift = pd.DataFrame(
    {
        'temporal (honest)': {k: v for k, v in scores[BEST].items() if k != 'expected'},
        'stratified (separability)': score(
            s_test[LABEL_COLUMN].astype(str).tolist(),
            refit.predict(s_test[TEXT_COLUMN].astype(str).tolist()),
        ),
    }
)
drift.loc[['macro F1', 'balanced accuracy', 'accuracy']]

## 8. Conclusion

*Fill in from the run. The structure of the argument:*

**What ships.** On short, templated, domain-specific ticket text the expected
result is that a calibrated linear model over word+char TF-IDF lands within a
couple of macro-F1 points of anything heavier, trains in seconds on CPU, and
can be asked which n-grams drove a class. **The LLM has to beat that by enough
to pay for itself** at the per-1k-ticket cost in section 4 - and to beat it on
the coverage curve, not only on macro-F1, because auto-routing is the product.

Where an LLM genuinely wins and this table will not show it: **the cold start.**
With no labelled history the linear arms cannot be trained at all, and the LLM
is at full strength on day one. For a greenfield deployment that is the
argument to weigh, not the macro-F1 delta.

**Threats to validity:**

* The dataset injects 7% label noise; no model can exceed that ceiling, and one
  that appears to is memorising the noise.
* The LLM ran on `LLM_TEST_CAP` tickets - smaller sample, wider error bars.
* One prompt was tested. Prompt wording is usually a bigger lever than model
  choice; sweeping it is the obvious next experiment.
* Anonymisation (notebook 1) replaces names with tags before this runs. That
  costs the classifier nothing here, but it is measured, not assumed.

In [ ]:
settings.ensure_dirs()
path = settings.reports_dir / 'experiment_03_classification.json'
path.write_text(
    json.dumps(
        {
            'text_column': TEXT_COLUMN,
            'split': 'temporal',
            'n_train': len(x_train),
            'n_test': len(x_test),
            'llm_ran': RUN_LLM,
            'offline': scores,
            'like_for_like_subset': subset,
            'best': BEST,
            'coverage_curve': curve.to_dict(orient='records'),
            'drift': drift.to_dict(),
        },
        indent=2,
        default=str,
    ),
    encoding='utf-8',
)
print('written to', path)